In [ ]:
# Instalaivimas reikalingų paketų
#!pip install git+https://github.com/huggingface/transformers accelerate
print("Instaliavimas baigtas")

In [ ]:
# Perspėjimų ignoravimas
import warnings
warnings.filterwarnings("ignore")  # Ignore all warnings

from transformers import AutoProcessor, AutoModelForVision2Seq
from PIL import Image
import requests
import torch
import os
import re
import pandas as pd
print("Importavimas baigtas")

In [ ]:
# Modelių naudojimas

device = "cuda" if torch.cuda.is_available() else "cpu"
checkpoint = "HuggingFaceM4/idefics2-8b"
processor = AutoProcessor.from_pretrained(checkpoint)
model = AutoModelForVision2Seq.from_pretrained(checkpoint, torch_dtype=torch.bfloat16).to(device)

print("Modelio krovimas baigtas")

In [ ]:
# Lentelės užkrovimas, generavimas ir saugojimas
df = pd.read_csv('flicker8k_edited.csv', sep=';')
import numpy as np

# Išsivalome lentelę
columns_to_clear = [
    'BLEU', 'ROGUE', 'METEOR',
    'BERTscore Precision', 'BERTscore Recall', 'BERTscore F1', 'SentenceBERT'
]
for col in columns_to_clear:
    if col in df.columns:
        df[col] = df[col].apply(lambda x: '')

image_folder = "Flickr8k"

#for index, row in df.head(20).iterrows():
for index, row in df.iterrows():
    filename = row["Image Name"]
    image_path = os.path.join(image_folder, filename)
    try:
        image = Image.open(image_path)
        prompt = "<image>\nUser: Keep it short. No more than 10 words. The text is the caption for the image. Do not write any additional text.\nAssistant:"
        inputs = processor(text=[prompt], images=[[image]], return_tensors="pt").to(device)
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=50,   # Short, single answer
            temperature=0.7,
            do_sample=True,
            top_p=0.9
        )

        generated_caption = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
        
        match = re.search(r'Assistant:\s*(.*)$', generated_caption)
        if match:
            generated_caption = match.group(1).strip()
        else:
            print("No caption found.")

        # Įrašymas į failą
        df.at[index, "Generated Caption"] = generated_caption

    except Exception as e:
        print(f"Error processing {filename}: {e}")
        df.at[index, "Generated Caption"] = f"Error: {e}"

# Saugojimas
df.to_csv("Results/flicker8k_en_20.csv", sep=';', index=False)
print("Generavimas baigtas")